In [ ]:
# ⚙️ 1. Könyvtárak telepítése
!pip install torch torchvision Pillow tqdm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 110.8 MB/s eta 0:00:00


In [ ]:
# 📂 1. Drive csatolása (ha még nem volt)
from google.colab import drive
drive.mount('/content/drive')

# 🧭 2. Zip fájl elérési útja a Drive-on (változtasd meg, ha kell)
drive_zip_path = '/content/drive/MyDrive/diplomamunka/unsorted_db.zip'

# 🎯 3. Célútvonal a VM-en (Colab lokális gép)
vm_zip_path = '/content/unsorted_db.zip'

# 🚀 4. Másolás a VM-re
import shutil
print("ZIP fájl másolása a VM-re, ez pár perc lehet nagy fájlnál...")
shutil.copyfile(drive_zip_path, vm_zip_path)
print("Kész! A fájl most már a Colab gépen van.")


Mounted at /content/drive
ZIP fájl másolása a VM-re, ez pár perc lehet nagy fájlnál...
Kész! A fájl most már a Colab gépen van.


In [ ]:
import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import os, shutil, zipfile
from tqdm import tqdm
import requests

# ⚡ 1. GPU detektálás
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPU használatban: {device}')

# 🧠 2. Modell betöltés (Places365 ResNet50)
model_url = 'http://places2.csail.mit.edu/models_places365/resnet50_places365.pth.tar'
model_path = 'resnet50_places365.pth.tar'

if not os.path.exists(model_path):
    print("Modell letöltése...")
    !wget -q $model_url -O $model_path

model = models.resnet50(num_classes=365)
checkpoint = torch.load(model_path, map_location=device)
state_dict = {k.replace('module.', ''): v for k, v in checkpoint['state_dict'].items()}
model.load_state_dict(state_dict)
model.eval().to(device)
print("Modell betöltve.")

# 📜 3. Kategóriák letöltése
cat_url = 'https://raw.githubusercontent.com/csailvision/places365/master/categories_places365.txt'
categories = requests.get(cat_url).text.strip().split('\n')
categories = [line.split(' ')[0][3:] for line in categories]

# 🎯 4. Előfeldolgozás
def transform_image(img):
    if img.size != (224, 224):
        img = img.resize((224, 224))
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    return transform(img)

# 🔍 5. Képosztályozás
def classify_image(img_path):
    try:
        img = Image.open(img_path).convert('RGB')
        input_tensor = transform_image(img).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(input_tensor)
            pred = output.argmax().item()
        return categories[pred]
    except Exception as e:
        print(f'HIBA: {img_path} – {e}')
        return 'unknown'

# 🗂️ 6. Kicsomagolás feltöltés után
uploaded_zip = '/content/unsorted_db.zip'  # <- Itt legyen a feltöltött ZIP fájl neve
extract_dir = '/content/unsorted_db'

if not os.path.exists(extract_dir):
    print("ZIP fájl kicsomagolása...")
    with zipfile.ZipFile(uploaded_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Kicsomagolás kész.")

# 🔎 7. Képek összegyűjtése
valid_exts = ('.jpg', '.jpeg', '.png')
all_files = []
for root, _, files in os.walk(extract_dir):
    for file in files:
        if file.lower().endswith(valid_exts):
            all_files.append(os.path.join(root, file))

print(f'Feldolgozandó képek száma: {len(all_files)}')

# 📋 8. Osztályozás
classified = []
for img_path in tqdm(all_files, desc="📊 Képek osztályozása"):
    category = classify_image(img_path)
    if category != 'unknown':
        classified.append((img_path, category))

# 🧹 9. Kategóriákba rendezés
target_root = '/content/sorted_db'
os.makedirs(target_root, exist_ok=True)

for img_path, category in tqdm(classified, desc="💾 Képek áthelyezése"):
    target_dir = os.path.join(target_root, category)
    os.makedirs(target_dir, exist_ok=True)

    file_name = os.path.basename(img_path)
    new_path = os.path.join(target_dir, file_name)

    try:
        shutil.move(img_path, new_path)
    except Exception as e:
        print(f'Nem sikerült áthelyezni: {img_path} – {e}')

# 📦 10. Újratömörítés
zip_output = '/content/sorted_db.zip'
print("Kategorizált képek tömörítése...")

def zipdir(path, ziph):
    for root, _, files in os.walk(path):
        for file in files:
            ziph.write(os.path.join(root, file),
                       os.path.relpath(os.path.join(root, file), path))

with zipfile.ZipFile(zip_output, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipdir(target_root, zipf)

print(f'Tömörített fájl elkészült: {zip_output}')

GPU használatban: cuda
Modell letöltése...
Modell betöltve.
ZIP fájl kicsomagolása...
Kicsomagolás kész.
Feldolgozandó képek száma: 180751


📊 Képek osztályozása:  74%|███████▍  | 134185/180751 [19:28<08:00, 96.83it/s]

HIBA: /content/unsorted_db/unsorted_db/d41d8cd98f00b204e9800998ecf8427e.jpg – cannot identify image file '/content/unsorted_db/unsorted_db/d41d8cd98f00b204e9800998ecf8427e.jpg'


💾 Képek áthelyezése: 100%|██████████| 180750/180750 [00:06<00:00, 28347.82it/s]


Kategorizált képek tömörítése...
Tömörített fájl elkészült: /content/sorted_db.zip


In [ ]:
print("Hello!")

Hello!


In [ ]:
# 📂 Drive csatolása (ha még nem volt)
from google.colab import drive
drive.mount('/content/drive')

# 🎯 Forrás és cél elérési út
local_zip = '/content/sorted_db.zip'
drive_target = '/content/drive/MyDrive/diplomamunka/sorted_db.zip'

# 🚀 Másolás
import shutil
print("Fájl másolása Drive-ra...")
shutil.copyfile(local_zip, drive_target)
print("Kész! Most már letöltheted a Drive-ból.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fájl másolása Drive-ra...
Kész! Most már letöltheted a Drive-ból.


In [ ]:
# 🔄 Drive csatolása
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 📦 Forrás és cél
local_zip = '/content/sorted_db.zip'
drive_target = '/content/drive/MyDrive/diplomamunka/sorted_db.zip'

# 📊 Progress bar-ral másolás (blokkonként)
import os
from tqdm import tqdm

def copy_with_progress(src, dst, block_size=1024 * 1024):  # 1MB blokkok
    total_size = os.path.getsize(src)
    with open(src, 'rb') as fsrc, open(dst, 'wb') as fdst:
        with tqdm(total=total_size, unit='B', unit_scale=True, desc="📤 Másolás Drive-ra") as pbar:
            while True:
                buf = fsrc.read(block_size)
                if not buf:
                    break
                fdst.write(buf)
                pbar.update(len(buf))

# 🚀 Fájl másolása visszajelzéssel
copy_with_progress(local_zip, drive_target)

print("✅ Másolás befejezve! Most már letöltheted a Drive-ból.")


Mounted at /content/drive


📤 Másolás Drive-ra: 100%|██████████| 8.17G/8.17G [00:22<00:00, 370MB/s]

✅ Másolás befejezve! Most már letöltheted a Drive-ból.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
from tqdm import tqdm

# 📦 Forrás és cél
local_zip = '/content/sorted_db.zip'
drive_target = '/content/drive/MyDrive/sorted_db/sorted_db.zip'

def copy_with_progress_and_flush(src, dst, block_size=1024 * 1024):  # 1MB
    total_size = os.path.getsize(src)
    with open(src, 'rb') as fsrc, open(dst, 'wb') as fdst:
        with tqdm(total=total_size, unit='B', unit_scale=True, desc="📤 Másolás Drive-ra") as pbar:
            while True:
                buf = fsrc.read(block_size)
                if not buf:
                    break
                fdst.write(buf)
                fdst.flush()           # 💡 belső buffer flush
                os.fsync(fdst.fileno())  # 💡 biztosan kiíratjuk a fájlrendszerre
                pbar.update(len(buf))

copy_with_progress_and_flush(local_zip, drive_target)

print("✅ Kész! A fájl most már **valóban** megjelent a Drive-ban.")


Mounted at /content/drive


📤 Másolás Drive-ra: 100%|██████████| 8.17G/8.17G [00:21<00:00, 374MB/s]

✅ Kész! A fájl most már **valóban** megjelent a Drive-ban.
